# E4 NF Geometry Debug

Fast, single-condition debug notebook for the E4 tiny-MLP weight-space experiment.

It does not run LR tuning, downstream optimization, or E0-E3. It only trains one RealNVP NF on the E4 geometry objective and checks whether the held-out pullback geometry improves.


In [ ]:
from __future__ import annotations

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from IPython.display import Markdown, display

from post_train_research.loss_landscape_analysis.flow_preconditioning.optimization import (
    run_direct_curves_batched,
    run_flow_curves_batched,
)

from post_train_research.loss_landscape_analysis.flow_preconditioning.e4_debug import (
    E4DebugConfig,
    build_e4_debug_state,
    train_e4_debug_flow,
)

plt.rcParams['figure.dpi'] = 130

from post_train_research.loss_landscape_analysis.flow_preconditioning.toy_mlp import (
    pack_theta,
    unpack_theta,
)


## Hardcoded Debug Config

Defaults are intentionally small so the notebook can be rerun while debugging. Increase `FLOW_STEPS`, `FLOW_RANDOM_SAMPLES`, or eval sample counts only after the fast run gives a clear signal.


In [ ]:
RUN_LABEL = 'e4_nf_geometry_debug_seed0_rho1e2_fast'
DEVICE = 'cuda:0' if torch.cuda.is_available() else 'cpu'

DEBUG_CFG = E4DebugConfig(
    run_label=RUN_LABEL,
    artifact_root='artifacts/loss_landscape_analysis/flow_preconditioning/e4_debug',
    device=DEVICE,
    dtype='float32',
    seed=0,
    rho=1e-2,
    flow_steps=150,
    eval_every=15,
    flow_batch_size=8,
    flow_lr=1e-3,
    flow_grad_clip_norm=10.0,
    flow_num_layers=8,
    flow_hidden_dim=64,
    flow_network_depth=2,
    flow_log_scale_clamp=1.5,
    flow_random_samples=64,
    flow_trajectory_count=4,
    flow_trajectory_steps=8,
    heldout_geometry_samples=32,
    train_eval_samples=16,
    heldout_eval_samples=16,
)

DEBUG_CFG


## Build E4 Probe And Pools


In [ ]:
state = build_e4_debug_state(DEBUG_CFG)

print('output_dir:', state.output_dir)
print('device:', state.flow_pool.device)
print('flow_pool:', tuple(state.flow_pool.shape))
print('heldout_pool:', tuple(state.heldout_pool.shape))
print('train_eval_pool:', tuple(state.train_eval_pool.shape))
print('heldout_eval_pool:', tuple(state.heldout_eval_pool.shape))
print('probe output dim:', int(state.probe(state.flow_pool[0]).numel()))


## Train One NF And Evaluate Geometry During Training


In [ ]:
result = train_e4_debug_flow(state)
history = result.history
final_geometry = result.final_geometry

print('output_dir:', result.output_dir)
print('history rows:', len(history))
print('final geometry rows:', len(final_geometry))
display(history.tail(8))
display(final_geometry)


## Fast Sanity Checks


In [ ]:
step0 = history.iloc[0]
last = history.iloc[-1]

identity_train_abs = abs(float(step0['train_flow_R']) - float(step0['train_original_R']))
identity_heldout_abs = abs(float(step0['heldout_flow_R']) - float(step0['heldout_original_R']))
train_delta = float(last['train_R_delta'])
heldout_delta = float(last['heldout_R_delta'])
train_ratio = float(last['train_R_ratio'])
heldout_ratio = float(last['heldout_R_ratio'])

print(f"identity mismatch train_R={identity_train_abs:.6g} heldout_R={identity_heldout_abs:.6g}")
print(f"final train delta={train_delta:.6g} ratio={train_ratio:.4f}")
print(f"final heldout delta={heldout_delta:.6g} ratio={heldout_ratio:.4f}")

if identity_train_abs > 1e-3 or identity_heldout_abs > 1e-3:
    diagnosis = 'BUG: identity-initialized flow does not match original geometry. Check flow.inverse / probe_jacobians_for_flow wiring.'
elif heldout_delta < 0.0:
    diagnosis = 'OK: heldout geometry improved in this fast debug run.'
elif train_delta < 0.0 and heldout_delta >= 0.0:
    diagnosis = 'NF improves train-pool geometry but not heldout. This points to overfit / pool mismatch / objective generalization.'
else:
    diagnosis = 'NF does not even improve train-pool geometry. Debug optimizer, objective sign, flow capacity, or gradient path.'

Markdown(f"### Diagnosis\n{diagnosis}")


## Geometry Curves


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8), constrained_layout=True)

ax = axes[0, 0]
ax.plot(history['step'], history['train_flow_R'], label='train flow R')
ax.plot(history['step'], history['heldout_flow_R'], label='heldout flow R')
ax.axhline(float(history['train_original_R'].iloc[0]), linestyle='--', color='C0', alpha=0.5, label='train original R')
ax.axhline(float(history['heldout_original_R'].iloc[0]), linestyle='--', color='C1', alpha=0.5, label='heldout original R')
ax.set_xlabel('NF step')
ax.set_ylabel('global R, lower is better')
ax.set_yscale('log')
ax.grid(True, alpha=0.25)
ax.legend(fontsize=8)

ax = axes[0, 1]
plot_rows = history[history['step'] > 0]
ax.plot(plot_rows['step'], plot_rows['batch_loss'], label='train batch objective')
ax.set_xlabel('NF step')
ax.set_ylabel('batch objective')
ax.set_yscale('log')
ax.grid(True, alpha=0.25)
ax.legend(fontsize=8)

ax = axes[1, 0]
ax.plot(history['step'], history['train_flow_cond_median'], label='train flow cond median')
ax.plot(history['step'], history['heldout_flow_cond_median'], label='heldout flow cond median')
ax.set_xlabel('NF step')
ax.set_ylabel('pullback metric eig cond median')
ax.set_yscale('log')
ax.grid(True, alpha=0.25)
ax.legend(fontsize=8)

ax = axes[1, 1]
ax.plot(history['step'], history['train_flow_flow_displacement_median'], label='train displacement')
ax.plot(history['step'], history['heldout_flow_flow_displacement_median'], label='heldout displacement')
ax.set_xlabel('NF step')
ax.set_ylabel('median |flow(theta)-theta|')
ax.grid(True, alpha=0.25)
ax.legend(fontsize=8)

figure_path = Path(result.output_dir) / 'geometry_debug_curves.png'
fig.savefig(figure_path, dpi=180, bbox_inches='tight')
plt.show()
print('saved:', figure_path)


## Small Raw vs NF Optimization Comparison

This is a fixed-configuration sanity check, not LR tuning. It optimizes the same fresh E4 MLP starts directly in raw `theta` space and in NF coordinates `u = f(theta)` using the trained debug flow.


In [ ]:
OPT_COMPARE_STARTS = 8
OPT_COMPARE_STEPS = 150
OPT_COMPARE_OPTIMIZER = 'adam'
OPT_COMPARE_LR = 3e-3
OPT_COMPARE_SEED = 90_001

compare_starts = state.problem.sample_starts(OPT_COMPARE_STARTS, seed=OPT_COMPARE_SEED)
compare_lrs = torch.full(
    (OPT_COMPARE_STARTS,),
    float(OPT_COMPARE_LR),
    device=compare_starts.device,
    dtype=compare_starts.dtype,
)

raw_curves = run_direct_curves_batched(
    train_loss_fn=state.problem.train_loss,
    test_loss_fn=state.problem.test_loss,
    theta0_batch=compare_starts,
    optimizer_name=OPT_COMPARE_OPTIMIZER,
    lrs=compare_lrs,
    steps=OPT_COMPARE_STEPS,
)

nf_curves = run_flow_curves_batched(
    train_loss_fn=state.problem.train_loss,
    test_loss_fn=state.problem.test_loss,
    flow=result.flow,
    theta0_batch=compare_starts,
    optimizer_name=OPT_COMPARE_OPTIMIZER,
    lrs=compare_lrs,
    steps=OPT_COMPARE_STEPS,
)

curve_rows = []
summary_rows = []
for method, curves in [('raw', raw_curves), ('nf', nf_curves)]:
    for start_index, curve in enumerate(curves):
        for step, (train_loss, test_loss) in enumerate(zip(curve.train_loss, curve.test_loss, strict=True)):
            curve_rows.append(
                {
                    'method': method,
                    'optimizer': OPT_COMPARE_OPTIMIZER,
                    'lr': OPT_COMPARE_LR,
                    'start_index': start_index,
                    'step': step,
                    'train_loss': float(train_loss),
                    'test_loss': float(test_loss),
                }
            )
        summary_rows.append(
            {
                'method': method,
                'optimizer': OPT_COMPARE_OPTIMIZER,
                'lr': OPT_COMPARE_LR,
                'start_index': start_index,
                'initial_train_loss': float(curve.train_loss[0]),
                'final_train_loss': float(curve.train_loss[-1]),
                'best_train_loss': float(curve.train_loss.min()),
                'initial_test_loss': float(curve.test_loss[0]),
                'final_test_loss': float(curve.test_loss[-1]),
                'best_test_loss': float(curve.test_loss.min()),
            }
        )

optimization_curves = pd.DataFrame(curve_rows)
optimization_summary = pd.DataFrame(summary_rows)
optimization_curves.to_csv(Path(result.output_dir) / 'small_raw_vs_nf_optimization_curves.csv', index=False)
optimization_summary.to_csv(Path(result.output_dir) / 'small_raw_vs_nf_optimization_summary.csv', index=False)

aggregate_summary = optimization_summary.groupby('method').agg(
    final_train_median=('final_train_loss', 'median'),
    best_train_median=('best_train_loss', 'median'),
    final_test_median=('final_test_loss', 'median'),
    best_test_median=('best_test_loss', 'median'),
).reset_index()

display(aggregate_summary)
display(optimization_summary.sort_values(['start_index', 'method']))
print('saved:', Path(result.output_dir) / 'small_raw_vs_nf_optimization_curves.csv')
print('saved:', Path(result.output_dir) / 'small_raw_vs_nf_optimization_summary.csv')


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4), constrained_layout=True)

for ax, loss_col, title in [
    (axes[0], 'train_loss', 'train loss'),
    (axes[1], 'test_loss', 'test loss'),
]:
    for method, frame in optimization_curves.groupby('method'):
        grouped = frame.groupby('step')[loss_col]
        median = grouped.median()
        q25 = grouped.quantile(0.25)
        q75 = grouped.quantile(0.75)
        ax.plot(median.index, median.values, label=method)
        ax.fill_between(median.index, q25.values, q75.values, alpha=0.12)
    ax.set_yscale('log')
    ax.set_xlabel('optimization step')
    ax.set_ylabel(title)
    ax.set_title(f'{title}: raw theta vs trained NF coordinates')
    ax.grid(True, alpha=0.25)
    ax.legend(fontsize=8)

opt_figure_path = Path(result.output_dir) / 'small_raw_vs_nf_optimization.png'
fig.savefig(opt_figure_path, dpi=180, bbox_inches='tight')
plt.show()
print('saved:', opt_figure_path)


## Loss Landscape: Raw Theta vs NF Coordinates

Same protocol as `mnist_one_layer_vit_loss_landscape_latent_vs_raw.ipynb`: build normalized 2D direction pairs, evaluate `L(alpha,beta)` on a square grid, plot `L(alpha,beta) - L(0,0)`, and compute `S_rho`, `M_rho`, and good-area metrics. Raw E4 directions are filter-wise normalized for `W1/W2` with biases fixed; NF directions are norm-matched in `u = f(theta)` space and decoded with `theta = f^{-1}(u)`.


In [ ]:
import time
from dataclasses import dataclass
from typing import Any, Iterable

LANDSCAPE_NUM_DIRECTIONS = 1
LANDSCAPE_GRID_POINTS = 31
LANDSCAPE_RHO_MAX = 1.0
LANDSCAPE_SEED = 123_456
LANDSCAPE_RHO_VALUES = (0.25, 0.5, 0.75, 1.0)
LANDSCAPE_TAU_VALUES = (1e-3, 1e-2, 5e-2)
LANDSCAPE_CENTER_MODE = 'best_raw_final'  # 'best_raw_final' or 'initial_start_0'

if LANDSCAPE_CENTER_MODE == 'best_raw_final':
    best_raw_row = optimization_summary[optimization_summary['method'] == 'raw'].sort_values('best_train_loss').iloc[0]
    center_start_index = int(best_raw_row['start_index'])
    theta_center = torch.as_tensor(
        raw_curves[center_start_index].final_theta,
        device=state.flow_pool.device,
        dtype=state.flow_pool.dtype,
    )
elif LANDSCAPE_CENTER_MODE == 'initial_start_0':
    center_start_index = 0
    theta_center = compare_starts[0].detach().clone()
else:
    raise ValueError(f'unknown LANDSCAPE_CENTER_MODE={LANDSCAPE_CENTER_MODE!r}')

result.flow.eval()
with torch.no_grad():
    base_loss = float(state.problem.train_loss(theta_center).detach().cpu().item())
    base_test_loss = float(state.problem.test_loss(theta_center).detach().cpu().item())
    u_center = result.flow(theta_center.reshape(1, -1))[0].reshape(-1).detach()

print('landscape center mode:', LANDSCAPE_CENTER_MODE)
print('center start index:', center_start_index)
print('base train loss:', base_loss)
print('base test loss:', base_test_loss)
print('u_center norm:', float(u_center.norm().detach().cpu().item()))


def _rowwise_norm_matched_direction(value: torch.Tensor, *, generator: torch.Generator) -> torch.Tensor:
    noise = torch.randn(value.shape, generator=generator, dtype=value.dtype, device='cpu').to(value.device)
    flat_value = value.reshape(value.shape[0], -1)
    flat_noise = noise.reshape(noise.shape[0], -1)
    value_norm = flat_value.norm(dim=1, keepdim=True).clamp_min(1e-12)
    noise_norm = flat_noise.norm(dim=1, keepdim=True).clamp_min(1e-12)
    return (flat_noise / noise_norm * value_norm).reshape_as(value)


def e4_filterwise_normalized_flat_direction(theta: torch.Tensor, *, generator: torch.Generator) -> torch.Tensor:
    w1, b1, w2, b2 = unpack_theta(theta.detach())
    d_w1 = _rowwise_norm_matched_direction(w1, generator=generator)
    d_w2 = _rowwise_norm_matched_direction(w2, generator=generator)
    return pack_theta(d_w1, torch.zeros_like(b1), d_w2, torch.zeros_like(b2)).detach().clone()


def make_raw_direction_pairs(theta: torch.Tensor, num_pairs: int, *, seed: int):
    generator = torch.Generator(device='cpu')
    generator.manual_seed(int(seed))
    return [
        (
            e4_filterwise_normalized_flat_direction(theta, generator=generator),
            e4_filterwise_normalized_flat_direction(theta, generator=generator),
        )
        for _ in range(int(num_pairs))
    ]


def make_nf_direction(u: torch.Tensor, generator: torch.Generator) -> torch.Tensor:
    noise = torch.randn(u.shape, generator=generator, dtype=u.dtype, device='cpu').to(u.device)
    return (noise / noise.norm().clamp_min(1e-12) * u.detach().norm().clamp_min(1e-12)).detach().clone()


def make_nf_direction_pairs(u: torch.Tensor, num_pairs: int, *, seed: int):
    generator = torch.Generator(device='cpu')
    generator.manual_seed(int(seed))
    return [(make_nf_direction(u, generator), make_nf_direction(u, generator)) for _ in range(int(num_pairs))]


@dataclass
class LandscapeResult:
    kind: str
    direction_idx: int
    alphas: np.ndarray
    betas: np.ndarray
    losses: np.ndarray
    base_loss: float


def evaluate_landscape(
    *,
    kind: str,
    direction_pairs: list[Any],
    base_loss: float,
    eval_fn,
    grid_points: int,
    rho_max: float,
) -> list[LandscapeResult]:
    alphas = np.linspace(-float(rho_max), float(rho_max), int(grid_points))
    betas = np.linspace(-float(rho_max), float(rho_max), int(grid_points))
    results = []
    for direction_idx, pair in enumerate(direction_pairs):
        losses = np.full((len(alphas), len(betas)), np.nan, dtype=np.float64)
        t0 = time.time()
        for i, alpha in enumerate(alphas):
            for j, beta in enumerate(betas):
                losses[i, j] = float(eval_fn(pair, float(alpha), float(beta)))
        item = LandscapeResult(
            kind=kind,
            direction_idx=direction_idx,
            alphas=alphas,
            betas=betas,
            losses=losses,
            base_loss=float(base_loss),
        )
        results.append(item)
        print(
            f'{kind} direction={direction_idx} done in {time.time() - t0:.1f}s '
            f'min_delta={np.nanmin(losses - base_loss):.4f} max_delta={np.nanmax(losses - base_loss):.4f}'
        )
    return results


def compute_landscape_metrics(
    results: list[LandscapeResult],
    rho_values: Iterable[float],
    tau_values: Iterable[float],
) -> pd.DataFrame:
    rows = []
    for item in results:
        alpha_grid, beta_grid = np.meshgrid(item.alphas, item.betas, indexing='ij')
        delta = item.losses - item.base_loss
        step_alpha = float(abs(item.alphas[1] - item.alphas[0])) if len(item.alphas) > 1 else 1.0
        step_beta = float(abs(item.betas[1] - item.betas[0])) if len(item.betas) > 1 else 1.0
        cell_area = step_alpha * step_beta
        radius2 = alpha_grid * alpha_grid + beta_grid * beta_grid
        for rho in rho_values:
            disk = radius2 <= float(rho) ** 2 + 1e-12
            disk_delta = delta[disk]
            row = {
                'kind': item.kind,
                'direction_idx': item.direction_idx,
                'rho': float(rho),
                'S_rho': float(np.nanmax(disk_delta)),
                'M_rho': float(np.nanmean(disk_delta)),
                'disk_grid_points': int(disk.sum()),
            }
            for tau in tau_values:
                good = disk & (item.losses <= item.base_loss + float(tau))
                row[f'A_tau={tau:g}_rho'] = float(good.sum() * cell_area)
                row[f'A_frac_tau={tau:g}_rho'] = float(good.sum() / max(1, disk.sum()))
            rows.append(row)
    return pd.DataFrame(rows)


raw_direction_pairs = make_raw_direction_pairs(theta_center, LANDSCAPE_NUM_DIRECTIONS, seed=LANDSCAPE_SEED)
nf_direction_pairs = make_nf_direction_pairs(u_center, LANDSCAPE_NUM_DIRECTIONS, seed=LANDSCAPE_SEED + 1000)


def raw_eval_fn(pair, alpha: float, beta: float) -> float:
    d1, d2 = pair
    theta = theta_center + float(alpha) * d1 + float(beta) * d2
    with torch.no_grad():
        return float(state.problem.train_loss(theta).detach().cpu().item())


def nf_eval_fn(pair, alpha: float, beta: float) -> float:
    d1, d2 = pair
    u = u_center + float(alpha) * d1 + float(beta) * d2
    with torch.no_grad():
        theta = result.flow.inverse(u.reshape(1, -1))[0].reshape(-1)
        return float(state.problem.train_loss(theta).detach().cpu().item())


raw_landscape_results = evaluate_landscape(
    kind='raw_theta',
    direction_pairs=raw_direction_pairs,
    base_loss=base_loss,
    eval_fn=raw_eval_fn,
    grid_points=LANDSCAPE_GRID_POINTS,
    rho_max=LANDSCAPE_RHO_MAX,
)
nf_landscape_results = evaluate_landscape(
    kind='nf_u',
    direction_pairs=nf_direction_pairs,
    base_loss=base_loss,
    eval_fn=nf_eval_fn,
    grid_points=LANDSCAPE_GRID_POINTS,
    rho_max=LANDSCAPE_RHO_MAX,
)
landscape_results = raw_landscape_results + nf_landscape_results
landscape_metrics = compute_landscape_metrics(landscape_results, LANDSCAPE_RHO_VALUES, LANDSCAPE_TAU_VALUES)

display(landscape_metrics)
landscape_metrics.to_csv(Path(result.output_dir) / 'landscape_metrics_raw_vs_nf.csv', index=False)

save_payload = {}
for item in landscape_results:
    prefix = f'{item.kind}_direction_{item.direction_idx}'
    save_payload[f'{prefix}_alphas'] = item.alphas
    save_payload[f'{prefix}_betas'] = item.betas
    save_payload[f'{prefix}_losses'] = item.losses
    save_payload[f'{prefix}_base_loss'] = np.array([item.base_loss], dtype=np.float64)
np.savez_compressed(Path(result.output_dir) / 'landscape_results_raw_vs_nf.npz', **save_payload)
print('saved:', Path(result.output_dir) / 'landscape_metrics_raw_vs_nf.csv')
print('saved:', Path(result.output_dir) / 'landscape_results_raw_vs_nf.npz')


In [ ]:
def plot_landscape(
    item: LandscapeResult,
    title: str | None = None,
    save_path: Path | None = None,
) -> None:
    alpha_grid, beta_grid = np.meshgrid(item.alphas, item.betas, indexing='ij')
    delta = item.losses - item.base_loss
    plt.figure(figsize=(5, 4))
    contour = plt.contourf(alpha_grid, beta_grid, delta, levels=40, cmap='magma')
    plt.colorbar(contour, label='L(alpha,beta) - L(0,0)')
    plt.contour(alpha_grid, beta_grid, delta, colors='black', linewidths=0.35, levels=12, alpha=0.55)
    plt.scatter([0], [0], c='cyan', s=20, edgecolors='black', linewidths=0.4, label='center')
    plt.xlabel('alpha')
    plt.ylabel('beta')
    plt.title(title or f'{item.kind} direction {item.direction_idx}')
    plt.legend(loc='best', fontsize=8, framealpha=0.85)
    plt.tight_layout()
    if save_path is not None:
        save_path.parent.mkdir(parents=True, exist_ok=True)
        plt.savefig(save_path, dpi=180, bbox_inches='tight')
    plt.show()


for item in landscape_results:
    plot_landscape(
        item,
        title=f'E4 {item.kind} loss landscape: direction {item.direction_idx}',
        save_path=Path(result.output_dir) / f'landscape_{item.kind}_direction_{item.direction_idx}.png',
    )

# One side-by-side figure for fast visual comparison of direction 0.
raw0 = raw_landscape_results[0]
nf0 = nf_landscape_results[0]
fig, axes = plt.subplots(1, 2, figsize=(11, 4), constrained_layout=True)
for ax, item, title in [
    (axes[0], raw0, 'raw theta'),
    (axes[1], nf0, 'trained NF u'),
]:
    alpha_grid, beta_grid = np.meshgrid(item.alphas, item.betas, indexing='ij')
    delta = item.losses - item.base_loss
    contour = ax.contourf(alpha_grid, beta_grid, delta, levels=40, cmap='magma')
    ax.contour(alpha_grid, beta_grid, delta, colors='black', linewidths=0.35, levels=12, alpha=0.55)
    ax.scatter([0], [0], c='cyan', s=20, edgecolors='black', linewidths=0.4)
    ax.set_title(title)
    ax.set_xlabel('alpha')
    ax.set_ylabel('beta')
fig.colorbar(contour, ax=axes, label='L(alpha,beta) - L(0,0)')
comparison_path = Path(result.output_dir) / 'landscape_raw_vs_nf_direction_0.png'
fig.savefig(comparison_path, dpi=180, bbox_inches='tight')
plt.show()
print('saved:', comparison_path)


## Final Compact Table


In [ ]:
cols = [
    'step',
    'batch_loss',
    'train_original_R',
    'train_flow_R',
    'train_R_delta',
    'train_R_ratio',
    'heldout_original_R',
    'heldout_flow_R',
    'heldout_R_delta',
    'heldout_R_ratio',
    'train_flow_trace_cv',
    'heldout_flow_trace_cv',
    'train_flow_cond_median',
    'heldout_flow_cond_median',
    'train_flow_flow_cond_median',
    'heldout_flow_flow_cond_median',
]
compact = history[cols].copy()
display(compact)
compact.to_csv(Path(result.output_dir) / 'compact_history.csv', index=False)
print('saved:', Path(result.output_dir) / 'compact_history.csv')


## Files Written


In [ ]:
for path in sorted(Path(result.output_dir).iterdir()):
    print(path.name, path.stat().st_size)
